In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "enviroment_bj").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("project_root:", ROOT)

project_root: C:\Users\alejo\OneDrive\Escritorio\Pablo\Profesional\Modelaje\Modelos de DL\blackjack-rl


In [3]:
from pprint import pprint

import torch

from enviroment_bj import BlackjackConfig, BlackjackEnvironment, ObservationConfig, ACTION_ORDER
from model.encoder import BlackjackObservationEncoder
from model.encoder.constants import HAND_SETTLEMENT_VALUES, PROGRESS_BUCKET_VALUES, PUBLIC_ACTION_TOKENS


In [4]:
def make_env(profile, shoe, seed=11, **config_overrides):
    obs = ObservationConfig.for_profile(profile)
    config = BlackjackConfig(
        n_decks=1,
        shoe_penetration=1.0,
        observation=obs,
        **config_overrides,
    )
    env = BlackjackEnvironment(config=config, seed=seed)
    env.load_shoe(shoe, total_cards=len(shoe))
    return env


def inspect_encoded_response(label, response, encoder):
    encoded = encoder(response)

    print("=" * 100)
    print("LABEL:", label)
    print("PROFILE:", encoder.config.profile)
    print("ACTION_ORDER:", ACTION_ORDER)
    print("observation keys:", sorted(response["observation"].keys()))
    print("table_rules keys:", sorted(response["table_rules"].keys()))
    print("action_mask:", response["action_mask"])
    print("action_mask tensor shape:", tuple(encoded["action_mask"].shape))
    print("state_vector shape:", tuple(encoded["state_vector"].shape))
    print("state_dim from encoder:", encoder.state_dim)
    print()

    print("module_dims:")
    pprint(dict(encoder.module_dims))
    print()

    print("module_slices:")
    pprint(dict(encoded["metadata"]["module_slices"]))
    print()

    print("module tensor shapes:")
    pprint({k: tuple(v.shape) for k, v in encoded["module_tensors"].items()})
    print()

    print("nonzero por modulo:")
    pprint({k: int(torch.count_nonzero(v)) for k, v in encoded["module_tensors"].items()})

    return encoded


In [5]:
env = make_env(
    "table_realistic_default",
    ["10", "6", "7", "10", "10", "9", "5", "2", "10", "K", "8"],
)

encoder = BlackjackObservationEncoder.from_profile("table_realistic_default")

betting = env.reset()
playing = env.step("bet_1x")
finished = env.step("stand")
next_round = env.reset()

print("BETTING phase:", betting["observation"]["decision_phase"])
print("PLAYING phase:", playing["observation"]["decision_phase"])
print("FINISHED done:", finished["done"])
print("NEXT ROUND phase:", next_round["observation"]["decision_phase"])


BETTING phase: betting
PLAYING phase: playing
FINISHED done: True
NEXT ROUND phase: betting


In [6]:
for label, response in [
    ("betting", betting),
    ("playing", playing),
    ("finished", finished),
    ("next_round", next_round),
]:
    print("\n" + "=" * 80)
    print(label)
    pprint(response["observation"]["temporal_context"])



betting
{'dealer_hands_seen_since_shuffle': 0,
 'dealer_hands_seen_total': 0,
 'estimated_shoe_progress': {'bucket': 'early', 'fraction_used': 0.0},
 'hands_since_observed_shuffle': None,
 'has_observed_shuffle_reference': False,
 'high_fraction_since_shuffle': 0.0,
 'high_minus_low_balance': 0.0,
 'low_fraction_since_shuffle': 0.0,
 'observed_cards_since_shuffle': 0,
 'observed_shuffle_reset': False,
 'player_hands_seen_since_shuffle': 0,
 'player_hands_seen_total': 0,
 'rounds_played_total': 1,
 'rounds_since_shuffle': 1,
 'shuffle_count': 2}

playing
{'dealer_hands_seen_since_shuffle': 1,
 'dealer_hands_seen_total': 1,
 'estimated_shoe_progress': {'bucket': 'mid',
                             'fraction_used': 0.36363636363636365},
 'hands_since_observed_shuffle': None,
 'has_observed_shuffle_reference': False,
 'high_fraction_since_shuffle': 0.0,
 'high_minus_low_balance': 0.0,
 'low_fraction_since_shuffle': 0.0,
 'observed_cards_since_shuffle': 0,
 'observed_shuffle_reset': False,

In [7]:
env = make_env(
    "table_realistic_default",
    ["10", "6", "7", "10", "10", "9", "5", "2", "10", "K", "8"],
    visible_shoe_change=True,
)

env.mark_observed_shuffle_reset()

betting = env.reset()
playing = env.step("bet_1x")
finished = env.step("stand")
next_round = env.reset()

for label, response in [
    ("betting", betting),
    ("playing", playing),
    ("finished", finished),
    ("next_round", next_round),
]:
    print("\n" + "=" * 80)
    print(label)
    pprint(response["observation"]["temporal_context"])



betting
{'dealer_hands_seen_since_shuffle': 0,
 'dealer_hands_seen_total': 0,
 'estimated_shoe_progress': {'bucket': 'early', 'fraction_used': 0.0},
 'hands_since_observed_shuffle': 0,
 'has_observed_shuffle_reference': True,
 'high_fraction_since_shuffle': 0.0,
 'high_minus_low_balance': 0.0,
 'low_fraction_since_shuffle': 0.0,
 'observed_cards_since_shuffle': 0,
 'observed_shuffle_reset': True,
 'player_hands_seen_since_shuffle': 0,
 'player_hands_seen_total': 0,
 'rounds_played_total': 1,
 'rounds_since_shuffle': 1,
 'shuffle_count': 2}

playing
{'dealer_hands_seen_since_shuffle': 1,
 'dealer_hands_seen_total': 1,
 'estimated_shoe_progress': {'bucket': 'mid',
                             'fraction_used': 0.36363636363636365},
 'hands_since_observed_shuffle': 0,
 'has_observed_shuffle_reference': True,
 'high_fraction_since_shuffle': 0.3333333333333333,
 'high_minus_low_balance': 0.0,
 'low_fraction_since_shuffle': 0.3333333333333333,
 'observed_cards_since_shuffle': 3,
 'observed_s

In [8]:
encoder = BlackjackObservationEncoder.from_profile("table_realistic_default")

encoded_playing = encoder(playing)
encoded_finished = encoder(finished)
encoded_next_round = encoder(next_round)

print("temporal dim:", encoder.module_dims["temporal"])
print("temporal tensor playing:")
print(encoded_playing["module_tensors"]["temporal"])

print("\ntemporal tensor finished:")
print(encoded_finished["module_tensors"]["temporal"])

print("\ntemporal tensor next_round:")
print(encoded_next_round["module_tensors"]["temporal"])


temporal dim: 29
temporal tensor playing:
tensor([0.0200, 0.0010, 0.0100, 0.0050, 0.0050, 0.0010, 0.0010, 0.3636, 0.0000,
        1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0072, 0.3333,
        0.3333, 0.0000])

temporal tensor finished:
tensor([0.0200, 0.0010, 0.0100, 0.0050, 0.0050, 0.0010, 0.0010, 0.4545, 0.0000,
        1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0120, 0.2000,
        0.6000, 0.4000])

temporal tensor next_round:
tensor([0.0200, 0.0020, 0.0200, 0.0050, 0.0050, 0.0010, 0.0010, 0.4545, 0.0000,
        1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0100, 0.0120, 0.2000,
        0.6000, 0.4000])


In [9]:
TEMPORAL_BLOCKS = {
    "numeric": (0, 7),
    "estimated_progress": (7, 11),
    "last_round_outcome": (11, 22),
    "observed_shuffle_block": (22, 29),
}

def inspect_temporal_blocks(encoded, label):
    t = encoded["module_tensors"]["temporal"]
    print("\n" + "=" * 80)
    print(label)
    for name, (start, end) in TEMPORAL_BLOCKS.items():
        print(f"{name:22s} slice=({start:2d}, {end:2d}) ->", t[start:end])

inspect_temporal_blocks(encoded_playing, "playing")
inspect_temporal_blocks(encoded_finished, "finished")
inspect_temporal_blocks(encoded_next_round, "next_round")



playing
numeric                slice=( 0,  7) -> tensor([0.0200, 0.0010, 0.0100, 0.0050, 0.0050, 0.0010, 0.0010])
estimated_progress     slice=( 7, 11) -> tensor([0.3636, 0.0000, 1.0000, 0.0000])
last_round_outcome     slice=(11, 22) -> tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
observed_shuffle_block slice=(22, 29) -> tensor([0.0000, 1.0000, 0.0000, 0.0072, 0.3333, 0.3333, 0.0000])

finished
numeric                slice=( 0,  7) -> tensor([0.0200, 0.0010, 0.0100, 0.0050, 0.0050, 0.0010, 0.0010])
estimated_progress     slice=( 7, 11) -> tensor([0.4545, 0.0000, 1.0000, 0.0000])
last_round_outcome     slice=(11, 22) -> tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
observed_shuffle_block slice=(22, 29) -> tensor([0.0000, 1.0000, 0.0000, 0.0120, 0.2000, 0.6000, 0.4000])

next_round
numeric                slice=( 0,  7) -> tensor([0.0200, 0.0020, 0.0200, 0.0050, 0.0050, 0.0010, 0.0010])
estimated_progress     slice=( 7, 11) -> tensor([0.4545, 0.0000, 1.0000, 0.0000])
last_

In [10]:
OBSERVED_SHUFFLE_FEATURE_NAMES = [
    "observed_shuffle_reset",
    "has_observed_shuffle_reference",
    "hands_since_observed_shuffle",
    "observed_cards_since_shuffle",
    "low_fraction_since_shuffle",
    "high_fraction_since_shuffle",
    "high_minus_low_balance",
]

def explain_observed_shuffle_features(response, encoded, label):
    raw = response["observation"]["temporal_context"]
    tensor_block = encoded["module_tensors"]["temporal"][22:29]

    print("\n" + "=" * 80)
    print(label)
    print("raw temporal_context values:")
    for name in OBSERVED_SHUFFLE_FEATURE_NAMES:
        print(f"  {name:30s}: {raw.get(name)}")

    print("\nencoded tensor block:")
    for name, value in zip(OBSERVED_SHUFFLE_FEATURE_NAMES, tensor_block):
        print(f"  {name:30s}: {float(value):.6f}")

explain_observed_shuffle_features(playing, encoded_playing, "playing")
explain_observed_shuffle_features(finished, encoded_finished, "finished")
explain_observed_shuffle_features(next_round, encoded_next_round, "next_round")



playing
raw temporal_context values:
  observed_shuffle_reset        : False
  has_observed_shuffle_reference: True
  hands_since_observed_shuffle  : 0
  observed_cards_since_shuffle  : 3
  low_fraction_since_shuffle    : 0.3333333333333333
  high_fraction_since_shuffle   : 0.3333333333333333
  high_minus_low_balance        : 0.0

encoded tensor block:
  observed_shuffle_reset        : 0.000000
  has_observed_shuffle_reference: 1.000000
  hands_since_observed_shuffle  : 0.000000
  observed_cards_since_shuffle  : 0.007212
  low_fraction_since_shuffle    : 0.333333
  high_fraction_since_shuffle   : 0.333333
  high_minus_low_balance        : 0.000000

finished
raw temporal_context values:
  observed_shuffle_reset        : False
  has_observed_shuffle_reference: True
  hands_since_observed_shuffle  : 0
  observed_cards_since_shuffle  : 5
  low_fraction_since_shuffle    : 0.2
  high_fraction_since_shuffle   : 0.6
  high_minus_low_balance        : 0.39999999999999997

encoded tensor block:


In [11]:
print("observed_history tensor:")
print(encoded_finished["module_tensors"]["observed_history"])

print("\ndiscard_summary tensor shape:")
print(tuple(encoded_finished["module_tensors"]["discard_summary"].shape))

print("\nraw observed_cards_history:")
pprint(finished["observation"]["observed_cards_history"])

print("\nraw discard_summary:")
pprint(finished["observation"]["discard_summary"])

print("\nraw temporal_context:")
pprint(finished["observation"]["temporal_context"])


observed_history tensor:
tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.2000, 0.2000, 0.0000, 0.0000,
        0.6000, 0.0000, 0.0000, 0.0000, 0.0120])

discard_summary tensor shape:
(144,)

raw observed_cards_history:
{'10': 3,
 '2': 0,
 '3': 0,
 '4': 0,
 '5': 0,
 '6': 1,
 '7': 1,
 '8': 0,
 '9': 0,
 'A': 0,
 'J': 0,
 'K': 0,
 'Q': 0}

raw discard_summary:
{'by_group': {'high': 3, 'low': 1, 'neutral': 1},
 'observed_cards_count': 5,
 'recent_cards': ['10', '6', '7', '10', '10']}

raw temporal_context:
{'dealer_hands_seen_since_shuffle': 1,
 'dealer_hands_seen_total': 1,
 'estimated_shoe_progress': {'bucket': 'mid',
                             'fraction_used': 0.45454545454545453},
 'hands_since_observed_shuffle': 0,
 'has_observed_shuffle_reference': True,
 'high_fraction_since_shuffle': 0.6,
 'high_minus_low_balance': 0.39999999999999997,
 'low_fraction_since_shuffle': 0.2,
 'observed_cards_since_shuffle': 5,
 'observed_shuffle_reset': False,
 'player_hands_seen_since_shuffle': 1,


In [12]:
env = make_env(
    "fully_observable_sim",
    ["10", "6", "7", "10", "10", "9", "5", "2", "10", "K", "8"],
    visible_shoe_change=True,
)

env.mark_observed_shuffle_reset()

betting = env.reset()
playing = env.step("bet_1x")
finished = env.step("stand")
next_round = env.reset()

encoder = BlackjackObservationEncoder.from_profile("fully_observable_sim")
encoded_full = encoder(next_round)

print("temporal_context:")
pprint(next_round["observation"]["temporal_context"])

print("\ntemporal shape:", tuple(encoded_full["module_tensors"]["temporal"].shape))
print("temporal nonzero:", int(torch.count_nonzero(encoded_full["module_tensors"]["temporal"])))

print("\nrecent_actions:")
pprint(next_round["observation"]["temporal_context"]["recent_actions"])


temporal_context:
{'dealer_hands_seen_since_shuffle': 1,
 'dealer_hands_seen_total': 1,
 'estimated_shoe_progress': {'bucket': 'mid',
                             'fraction_used': 0.45454545454545453},
 'hands_since_observed_shuffle': 1,
 'has_observed_shuffle_reference': True,
 'high_fraction_since_shuffle': 0.6,
 'high_minus_low_balance': 0.39999999999999997,
 'last_round_outcome': {'dealer_cards': ['6', '10', '10'],
                        'dealer_has_blackjack': False,
                        'dealer_total': 26,
                        'hand_rewards': [1.0],
                        'hand_settlements': ['win'],
                        'insurance_reward': 0.0,
                        'reward': 1.0,
                        'round_index': 1},
 'low_fraction_since_shuffle': 0.2,
 'observed_cards_since_shuffle': 5,
 'observed_shuffle_reset': False,
 'player_hands_seen_since_shuffle': 1,
 'player_hands_seen_total': 1,
 'recent_actions': [{'action': 'reset_to_betting',
                    

In [13]:
encoder = BlackjackObservationEncoder.from_profile("table_realistic_default")
encoded = encoder(finished)

print("state_vector shape:", tuple(encoded["state_vector"].shape))
print("action_mask shape:", tuple(encoded["action_mask"].shape))
print("module_dims:")
pprint(encoded["metadata"]["module_dims"])

print("\nmodule_slices:")
pprint(encoded["metadata"]["module_slices"])

print("\nfirst 80 values of state_vector:")
print(encoded["state_vector"][:80])


state_vector shape: (1111,)
action_mask shape: (10,)
module_dims:
{'bet': 1,
 'betting_context': 9,
 'discard_summary': 144,
 'hand': 185,
 'hand_context': 5,
 'insurance': 2,
 'observed_history': 14,
 'other_hands': 700,
 'rules': 22,
 'temporal': 29}

module_slices:
{'bet': (901, 902),
 'betting_context': (892, 901),
 'discard_summary': (938, 1082),
 'hand': (0, 185),
 'hand_context': (885, 890),
 'insurance': (890, 892),
 'observed_history': (924, 938),
 'other_hands': (185, 885),
 'rules': (902, 924),
 'temporal': (1082, 1111)}

first 80 values of state_vector:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.])


In [14]:
def summarize_modules(encoded):
    meta = encoded["metadata"]
    state = encoded["state_vector"]

    for name, (start, end) in meta["module_slices"].items():
        chunk = state[start:end]
        print(
            f"{name:20s} | slice=({start:4d}, {end:4d}) | "
            f"shape={tuple(chunk.shape)} | nonzero={int(torch.count_nonzero(chunk))}"
        )

summarize_modules(encoded_finished)


hand                 | slice=(   0,  185) | shape=(185,) | nonzero=2
other_hands          | slice=( 185,  885) | shape=(700,) | nonzero=8
hand_context         | slice=( 885,  890) | shape=(5,) | nonzero=1
insurance            | slice=( 890,  892) | shape=(2,) | nonzero=0
betting_context      | slice=( 892,  901) | shape=(9,) | nonzero=5
bet                  | slice=( 901,  902) | shape=(1,) | nonzero=0
rules                | slice=( 902,  924) | shape=(22,) | nonzero=10
observed_history     | slice=( 924,  938) | shape=(14,) | nonzero=4
discard_summary      | slice=( 938, 1082) | shape=(144,) | nonzero=14
temporal             | slice=(1082, 1111) | shape=(29,) | nonzero=14


In [15]:
FEATURE_NOTES = {
    "observed_cards_since_shuffle": "Cuántas cartas visibles se han observado desde el último shuffle observado.",
    "low_fraction_since_shuffle": "Fracción de cartas low (2-6) dentro de las cartas observadas desde ese shuffle.",
    "high_fraction_since_shuffle": "Fracción de cartas high (10-A) dentro de las cartas observadas desde ese shuffle.",
    "high_minus_low_balance": "high_fraction_since_shuffle - low_fraction_since_shuffle.",
}

pprint(FEATURE_NOTES)


{'high_fraction_since_shuffle': 'Fracción de cartas high (10-A) dentro de las '
                                'cartas observadas desde ese shuffle.',
 'high_minus_low_balance': 'high_fraction_since_shuffle - '
                           'low_fraction_since_shuffle.',
 'low_fraction_since_shuffle': 'Fracción de cartas low (2-6) dentro de las '
                               'cartas observadas desde ese shuffle.',
 'observed_cards_since_shuffle': 'Cuántas cartas visibles se han observado '
                                 'desde el último shuffle observado.'}
